## Week 05 Homework Submission

Build a trivia quiz agent, instrument it with monitoring, and analyze its behavior through traces. The tools code is provided - your focus is on the agent, monitoring, querying, and analysis.

The agent is a trivia quizmaster powered by the Open Trivia Database (https://opentdb.com/) - a free API for trivia questions, no authentication needed. You can explore the available categories at https://opentdb.com/api_config.php.

Here's how a session works:
* You pick a category and say "Let's play 3 medium questions from History"
* The agent fetches questions from the API and asks you the first one with multiple choice options
* You answer, and the agent tells you if you're right or wrong, then asks the next question
* After all questions, the agent gives your final score and explains the correct answers


Agent framework - PydanticAI
Observability platform - Logfire

In [1]:
from pydantic_ai import Agent

from trivia_tools import TriviaTools

tools = TriviaTools()
trivia_categories = tools.get_categories()
print(trivia_categories)
print()
print("Number of categories: ", len(trivia_categories.split('\n'))) # 24

9: General Knowledge
10: Entertainment: Books
11: Entertainment: Film
12: Entertainment: Music
13: Entertainment: Musicals & Theatres
14: Entertainment: Television
15: Entertainment: Video Games
16: Entertainment: Board Games
17: Science & Nature
18: Science: Computers
19: Science: Mathematics
20: Mythology
21: Sports
22: Geography
23: History
24: Politics
25: Art
26: Celebrities
27: Animals
28: Vehicles
29: Entertainment: Comics
30: Science: Gadgets
31: Entertainment: Japanese Anime & Manga
32: Entertainment: Cartoon & Animations

Number of categories:  24


### Q1. Create and run agent

In [2]:
instructions = """You are a trivia quizmaster. When asked to play trivia:
1. Use the available tools to fetch trivia questions
2. Ask the player one question at a time with multiple choice options
3. Wait for their answer before moving to the next question
4. When the player answers, explain why the correct answer is correct - add interesting context and facts
5. After all questions, give the final score
""".strip()

In [3]:
trivia_tools = TriviaTools()

agent = Agent(
    'openai:gpt-4o-mini',
    tools=[trivia_tools.get_categories, trivia_tools.get_questions],
    instructions=instructions,
)

What is the first tool the agent calls?

In [9]:
user_prompt = "Let's play a trivia game! Ask me 5 questions about general knowledge."

result = await agent.run(
    user_prompt
)

02:21:49.988 agent run
02:21:49.991   chat gpt-4o-mini
02:21:51.101   running tool: get_categories
02:21:52.337   chat gpt-4o-mini
02:21:53.442   running tool: get_questions
02:21:54.266   chat gpt-4o-mini


In [ ]:
from pydantic_ai.messages import ToolCallPart

# Check message history
first_tool = next(
    (part for msg in result.all_messages() for part in msg.parts if isinstance(part, ToolCallPart)),
    None
)
print("First tool called:", first_tool.tool_name)


First tool called: get_categories


In [11]:
list((part for msg in result.all_messages() for part in msg.parts if isinstance(part, ToolCallPart)))

[ToolCallPart(tool_name='get_categories', args='{}', tool_call_id='call_6TzoYjCnnsp2H6ms05iii6Mr'),
 ToolCallPart(tool_name='get_questions', args='{"amount":5,"category":9,"difficulty":"medium"}', tool_call_id='call_YCI6AoaBcoNcUay7PSSfBDhi')]

### Q2. Set up monitoring

Create a Logfire project at https://logfire.pydantic.dev/, get a write token, and configure it.

Run the agent once and check the Logfire dashboard.

What is the span_name for the top-level agent span?
* "agent run"

In [4]:
import logfire
from dotenv import load_dotenv

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()


Logfire project URL: 
https://logfire-us.pydantic.dev/doppel/trivia-quiz-ai-buildcamp


In [15]:
result = await agent.run(
    user_prompt
)

02:26:26.364 agent run
02:26:26.366   chat gpt-4o-mini
02:26:27.278   running tool: get_categories
02:26:28.874   chat gpt-4o-mini
02:26:29.865   running tool: get_questions
02:26:30.607   chat gpt-4o-mini


In [14]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content="Let's play a trivia game! Ask me 5 questions about general knowledge.", timestamp=datetime.datetime(2026, 5, 30, 18, 21, 56, 996557, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 30, 18, 21, 56, 996734, tzinfo=datetime.timezone.utc), instructions='You are a trivia quizmaster. When asked to play trivia:\n1. Use the available tools to fetch trivia questions\n2. Ask the player one question at a time with multiple choice options\n3. Wait for their answer before moving to the next question\n4. When the player answers, explain why the correct answer is correct - add interesting context and facts\n5. After all questions, give the final score', run_id='019e7a1f-1182-75b0-b7bc-8fb2d75ec3fb', conversation_id='019e7a1f-1182-75b0-b7bc-8fb163c2e661'),
 ModelResponse(parts=[ToolCallPart(tool_name='get_categories', args='{}', tool_call_id='call_hAL9JnYyKcaU3kMET8oMF1Ag')], usage=RequestUsage(input_tokens=197, output_tokens=10, detai

### Q3. Play a full session

In [7]:
## run function that handlesthe interactive conversation loop

import nest_asyncio
nest_asyncio.apply()

def run(prompt):
    message_history = []

    while True:
        result = agent.run_sync(prompt, message_history=message_history)
        print(result.output)
        message_history = result.all_messages()

        prompt = input("You (write 'stop' to stop): ")
        if not prompt or prompt.lower().strip() == 'stop':
            break

In [22]:
run("Let's play 5 easy questions from Science & Nature")

02:35:30.367 agent run
02:35:30.369   chat gpt-4o-mini
02:35:31.740   running tool: get_categories
02:35:32.709   chat gpt-4o-mini
02:35:33.681   running tool: get_questions
02:35:34.603   chat gpt-4o-mini
Great! Let's get started with your science and nature trivia. Here's your first question:

### Question 1:
The asteroid belt is located between which two planets?
- A) Jupiter and Saturn
- B) Mercury and Venus
- C) Mars and Jupiter
- D) Earth and Mars

What's your answer?
02:36:09.586 agent run
02:36:09.589   chat gpt-4o-mini
Correct! The asteroid belt is indeed located between **Mars and Jupiter**.

### Explanation:
The asteroid belt is a region in space filled with numerous rocky bodies known as asteroids. It lies between the orbits of Mars and Jupiter and is thought to be remnants of the early solar system that never coalesced into a planet due to the gravitational influence of nearby Jupiter. It contains a vast number of small objects, with the largest being the dwarf planet Cere

Play through the session - answer all 5 questions, then type "stop".

Look at the Logfire dashboard.

You'll see that each agent run shows up as a separate trace - they're not grouped together.

How many separate "agent run" traces do you see for this session?
* 6

### Q4. Grouping traces

***WANT:*** entire session grouped under one parent span

In [ ]:
with logfire.span('trivia_session'):
    run("Let's play 5 easy questions from Science & Nature")

02:42:10.781 trivia_session
02:42:10.784   agent run
02:42:10.785     chat gpt-4o-mini
02:42:11.855     running tool: get_categories
02:42:13.161     chat gpt-4o-mini
02:42:13.964     running tool: get_questions
02:42:14.941     chat gpt-4o-mini
Great! Let's get started with your trivia on Science & Nature. Here comes the first question:

**Question 1:** Rhinoplasty is a surgical procedure on what part of the human body?  
A) Ears  
B) Chin  
C) Neck  
D) Nose  

What's your answer?
02:42:27.552   agent run
02:42:27.554     chat gpt-4o-mini
Correct! Rhinoplasty is indeed a surgical procedure performed on the nose. 

Rhinoplasty can be done for various reasons, including cosmetic adjustments to improve appearance, or for functional purposes to correct breathing difficulties. It's one of the most common plastic surgeries performed worldwide. 

Ready for the next question? 

**Question 2:** Which element has the highest melting point?  
A) Tungsten  
B) Platinum  
C) Osmium  
D) Carbon  


All runs should now be nested under the "trivia_session" span.

What is the total input tokens of the session?

In [24]:
{
  "attributes": {
    "gen_ai.operation.name": "chat",
    "gen_ai.provider.name": "openai",
    "gen_ai.request.model": "gpt-4o-mini",
    "gen_ai.response.model": "gpt-4o-mini-2024-07-18",
    "gen_ai.system": "openai",
    "gen_ai.token.type": "input"
  },
  "total": 5530
}

{'attributes': {'gen_ai.operation.name': 'chat',
  'gen_ai.provider.name': 'openai',
  'gen_ai.request.model': 'gpt-4o-mini',
  'gen_ai.response.model': 'gpt-4o-mini-2024-07-18',
  'gen_ai.system': 'openai',
  'gen_ai.token.type': 'input'},
 'total': 5530}

### Q5. Get trace IDs with SQL

Access the run data programatically
* Create read token from Logfire project settings
* Set up query client from the downloading traces lesson

In [15]:
import dotenv
dotenv.load_dotenv()

import os
from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_READ_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)


What is the trace ID of your grouped session?

In [36]:
trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'trivia_session'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """
)

In [39]:
trace_id = trace_rows['rows'][0]['trace_id']

trace_id

'019e7a3196dd38df24ba6b24334dda85'

### Q6. Reconstruct runs and calculate costs

Use the trace_replay module from the downloading traces lesson to fetch one of your traces and convert it to an AgentRunResult.

Now calculate the approximate total cost of the session using gpt-4o-mini pricing ($0.15 per 1M input tokens, $0.60 per 1M output tokens). You can get the token counts from the query client or from the Logfire UI.

In [62]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def calculate_cost(model_name, input_tokens, output_tokens):
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost


In [ ]:
from trace_replay import fetch_trace, trace_to_run_result
trace = fetch_trace(trace_id, logfire_query_client)
trace

In [ ]:
run_result = trace_to_run_result(trace)
num_input_tokens = run_result._state.usage.input_tokens
num_output_tokens = run_result._state.usage.output_tokens

print("# of trace input tokens: ", num_input_tokens)
print("# of trace output tokens: ", num_output_tokens)

# of trace input tokens:  1173
# of trace output tokens:  123


In [64]:
calculate_cost("gpt-4o-mini", num_input_tokens, num_output_tokens)

0.00024974999999999997

### Bonus Q. Feedback tracking

Add thumbs up/down feedback tracking to agent runs, similar to what we did in the tracking user feedback lesson

In [10]:
import questionary

def ask_feedback_cli():
    result = questionary.select(
        "How was the trivia session?",
        choices=["👍 Good", "👎 Bad", "Skip"],
    ).ask()

    if result is None or result == "Skip":
        return None

    return 1 if "Good" in result else -1

def ask_feedback_jupyter():
    print("How was the trivia session?")
    print("1. 👍 Good")
    print("2. 👎 Bad")
    print("3. Skip")
    choice = input("Enter 1, 2, or 3: ")
    return {"1": 1, "2": -1, "3": None}.get(choice, None)



Call this function at the end of your session (after the run() loop finishes, but still inside the logfire.span() block) and send the result to Logfire using logfire.info().

In [13]:
with logfire.span('trivia_session_bonus'):
    session_context = logfire.get_context()
    run("Let's play 2 medium questions from Science & Nature")
    user_feedback = ask_feedback_jupyter()

    with logfire.attach_context(session_context):
        logfire.info("session_user_feedback", feedback=user_feedback)

18:20:36.864 trivia_session_bonus
18:20:36.866   agent run
18:20:36.868     chat gpt-4o-mini
18:20:38.090     running tool: get_categories
18:20:38.091     running tool: get_categories
18:20:39.272     chat gpt-4o-mini
18:20:40.558     running tool: get_questions
18:20:41.385     chat gpt-4o-mini
Great! Let's start with your first question.

**Question 1:** What mineral has the lowest number on the Mohs scale?
- A) Quartz
- B) Diamond
- C) Talc
- D) Gypsum

What's your answer?
18:20:53.432   agent run
18:20:53.434     chat gpt-4o-mini
Correct! The answer is **C) Talc**.

Talc is a mineral that is very soft and is assigned a value of 1 on the Mohs scale of mineral hardness. This scale ranges from 1 to 10, with diamond at the top of the scale with a value of 10, indicating its extreme hardness. Talc is commonly used in products like talcum powder, and its softness makes it easy to grind into a fine powder.

Now, onto your second question.

**Question 2:** Tetsuya Fujita was a scientist t

Run a few more sessions and record feedback for each. Then query Logfire to count positive vs negative feedback.

How do you make feedback events appear in the same trace as the agent run?
* By using logfire.attach_context() with the session context

In [ ]:
with logfire.span('trivia_session_bonus'):
    session_context = logfire.get_context()
    run("Let's play 2 medium questions from General Knowledge")
    user_feedback = ask_feedback_jupyter()

    with logfire.attach_context(session_context):
        logfire.info("session_user_feedback", feedback=user_feedback)

01:26:48.462 trivia_session_bonus
01:26:48.464   agent run
01:26:48.466     chat gpt-4o-mini
01:26:50.042     running tool: get_categories
01:26:51.909     chat gpt-4o-mini
01:26:53.062     running tool: get_questions
01:26:53.828     chat gpt-4o-mini
Here we go! Here's your first medium question from General Knowledge:

**Question 1:** What was Mountain Dew's original slogan?  
A) Give Me A Dew  
B) Do The Dew  
C) Get' that barefoot feelin' drinkin' Mountain Dew  
D) Yahoo! Mountain Dew... It'll tickle your innards!  

What’s your answer?
01:27:16.365   agent run
01:27:16.367     chat gpt-4o-mini
That's not correct! The correct answer is **D) Yahoo! Mountain Dew... It'll tickle your innards!**

This slogan was part of a more playful advertising campaign before Mountain Dew became the mainstream brand we know today. Mountain Dew was originally created as a mixer for whiskey in the 1940s, and over the years, it has transformed its image to appeal to a younger audience, incorporating en

In [ ]:
with logfire.span('trivia_session_bonus'):
    session_context = logfire.get_context()
    run("Let's play 2 medium questions from General Knowledge")
    user_feedback = ask_feedback_jupyter()

    with logfire.attach_context(session_context):
        logfire.info("session_user_feedback", feedback=user_feedback)

01:28:16.351 trivia_session_bonus
01:28:16.353   agent run
01:28:16.354     chat gpt-4o-mini
01:28:17.698     running tool: get_categories
01:28:17.699     running tool: get_questions
01:28:18.923     chat gpt-4o-mini
Great! Let's get started with your first question.

### Question 1: What is the Portuguese word for "Brazil"?
A) Brazil  
B) Brasil  
C) Brasilia  
D) Brasíl  

What's your answer?
01:28:30.397   agent run
01:28:30.399     chat gpt-4o-mini
That's not correct. The correct answer is **B) Brasil**.

### Explanation:
In Portuguese, the country is called "Brasil," which is how it's spelled and pronounced in the language. The name derives from the Brazilwood tree, which was collected by Portuguese explorers when they first arrived in the region. It's interesting to note that while "Brazil" is the English name, "Brasil" reflects the original language and is a crucial part of the country's identity.

Let's move on to the second question!

### Question 2: Out of these four buildin

In [ ]:
with logfire.span('trivia_session_bonus'):
    session_context = logfire.get_context()
    run("Let's play 2 medium questions from General Knowledge")
    user_feedback = ask_feedback_jupyter()

    with logfire.attach_context(session_context):
        logfire.info("session_user_feedback", feedback=user_feedback)

01:31:25.042 trivia_session_bonus
01:31:25.044   agent run
01:31:25.046     chat gpt-4o-mini
01:31:25.975     running tool: get_categories
01:31:27.431     chat gpt-4o-mini
01:31:28.363     running tool: get_questions
01:31:29.100     chat gpt-4o-mini
Let's start the trivia! Here's your first question:

**Question 1:** In which country was the 1992 Summer Olympics Games held?  
A) Russia  
B) Korea  
C) USA  
D) Spain  

What's your answer?
01:46:03.797   agent run
01:46:03.798     chat gpt-4o-mini
That's a great guess, but the correct answer is **D) Spain**. 

The 1992 Summer Olympics were held in Barcelona, Spain. This event was notable not only for its competitive spirit but also for the profound transformation it brought to the city, showcasing its culture and modern architecture. It marked the first time that the Olympics were held in Spain, and it was significant for featuring the "Dream Team," the United States men's basketball team that included superstar athletes such as Micha

In [35]:
trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'trivia_session_bonus'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """
)

In [36]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]#[0]['trace_id']
trace_ids

['019e7d8cc080976852227a48b372b000']

In [37]:
trace = fetch_trace('019e7d8cc080976852227a48b372b000', logfire_query_client)

In [38]:
trace

TraceData(all_messages=[{'role': 'user', 'parts': [{'type': 'text', 'content': "Let's play 2 medium questions from Science & Nature"}]}, {'role': 'assistant', 'parts': [{'type': 'tool_call', 'id': 'call_uxCDxqeBgtWlNLjZlEgQLIEg', 'name': 'get_categories', 'arguments': '{}'}, {'type': 'tool_call', 'id': 'call_BK6iQuKzfne9mynV66EP2Lfl', 'name': 'get_categories', 'arguments': '{}'}], 'finish_reason': 'tool_call'}, {'role': 'user', 'parts': [{'type': 'tool_call_response', 'id': 'call_uxCDxqeBgtWlNLjZlEgQLIEg', 'name': 'get_categories', 'result': '9: General Knowledge\n10: Entertainment: Books\n11: Entertainment: Film\n12: Entertainment: Music\n13: Entertainment: Musicals & Theatres\n14: Entertainment: Television\n15: Entertainment: Video Games\n16: Entertainment: Board Games\n17: Science & Nature\n18: Science: Computers\n19: Science: Mathematics\n20: Mythology\n21: Sports\n22: Geography\n23: History\n24: Politics\n25: Art\n26: Celebrities\n27: Animals\n28: Vehicles\n29: Entertainment: Comi

In [49]:
# Query records table by filtering by span_name = 'session_user_feedback' and extract the feedback attribute
# in Logfire's SQL (PostgreSQL compatible), attributes are accesssed with ->>

result = logfire_query_client.query_arrow("""
    SELECT 
        trace_id,
        start_timestamp,
        attributes->>'feedback' as feedback
    FROM records
    WHERE span_name = 'session_user_feedback'
    ORDER BY start_timestamp DESC
    LIMIT 4
""")

import pandas as pd
df = result.to_pandas()
print(df)


                           trace_id                  start_timestamp feedback
0  019e7f1729f2dee3e0ee03cdca105497 2026-05-31 17:46:33.009139+00:00       -1
1  019e7f1448df3a20f10e0bde789411cc 2026-05-31 17:29:12.133720+00:00      NaN
2  019e7f12f18e5cc906081e699009443d 2026-05-31 17:28:02.965394+00:00       -1
3  019e7d8cc080976852227a48b372b000 2026-05-31 10:21:44.271652+00:00        1
